In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import time
import numpy as np
from scipy.signal import butter, lfilter, hilbert, sosfilt
import mne
from scipy.io import loadmat


# modified feature extraction functions for single sample processing
def _extract_hga_lfs_direct(data, fs=1000, fs_feat=200):
    """Refer to _extract_hga_lfs"""
    n_samples, n_channels = data.shape
    T = fs_feat
    q = fs // fs_feat

    # HGA
    b_hga, a_hga = butter(4, [70 / (fs / 2), 200 / (fs / 2)], btype='bandpass')
    hga_filt = lfilter(b_hga, a_hga, data, axis=0)
    analytic = hilbert(hga_filt, axis=0)
    envelope = np.abs(analytic)
    # Binning
    hga_feat = envelope.reshape(T, q, n_channels).mean(axis=1).T

    # LFS
    b_lfs, a_lfs = butter(4, 100 / (fs / 2), btype='low')
    lfs_filt = lfilter(b_lfs, a_lfs, data, axis=0)
    # Decimation
    lfs_feat = lfs_filt[::q, :].T[:, :T]

    # Combine -> [n_channels, T, 2]
    return np.stack([hga_feat, lfs_feat], axis=-1).astype(np.float32)


def _extract_wavelet1_direct(data, fs=1000, fs_feat=10, freqs=np.linspace(10, 150, 15)):
    """Refer to _extract_wavelet"""
    n_samples, n_chans = data.shape
    T = fs_feat

    # Downsampling (1000 Hz -> 500 Hz)
    b_lp, a_lp = butter(4, 200 / (fs / 2), btype='low')
    data_ds = lfilter(b_lp, a_lp, data, axis=0)[::2, :]  # [500, C]
    fs_new = fs // 2

    epoch_data = data_ds.T[np.newaxis, ...]

    n_cycles = freqs / 5.0
    tfr = np.abs(mne.time_frequency.tfr_array_morlet(
        epoch_data, sfreq=fs_new, freqs=freqs, n_cycles=n_cycles, output='complex', n_jobs=1, verbose=False
    ))

    # Time binning
    sub = tfr.shape[-1] // T
    tfr_res = tfr[0].reshape(n_chans, len(freqs), T, sub).mean(axis=-1)

    # Transpose to [n_channels, T, n_freqs]
    return np.transpose(tfr_res, (0, 2, 1)).astype(np.float32)

def _extract_wavelet2_direct(data, fs=1000, fs_feat=100, freqs=np.geomspace(5, 195, 20)):
    """Refer to _extract_wavelet"""
    n_samples, n_chans = data.shape
    T = fs_feat

    # Downsampling (1000 -> 500)
    b_lp, a_lp = butter(4, 200 / (fs / 2), btype='low')
    data_ds = lfilter(b_lp, a_lp, data, axis=0)[::2, :]  # [500, C]
    fs_new = fs // 2

    epoch_data = data_ds.T[np.newaxis, ...]

    n_cycles = freqs / 5.0
    tfr = mne.time_frequency.tfr_array_morlet(
        epoch_data, sfreq=fs_new, freqs=freqs, n_cycles=n_cycles,
        output='power', n_jobs=1, verbose=False
    )

    # Time binning
    sub = tfr.shape[-1] // T
    tfr_res = tfr[0].reshape(n_chans, len(freqs), T, sub).mean(axis=-1)

    # Transpose to [n_channels, T, n_freqs]
    return np.transpose(tfr_res, (0, 2, 1)).astype(np.float32)

def _extract_physiologicalBand_direct(data, fs=1000, fs_feat=10):
    """Refer to _extract_physiologicalBand"""
    n_samples, n_chans = data.shape
    T = fs_feat
    q = fs // fs_feat

    bands = {"δ": (1.5, 5), "θ": (5, 8), "α": (8, 12), "β1": (12, 24), "β2": (24, 34), "γ1": (34, 60), "γ2": (60, 100),
             "γ3": (100, 130)}

    data_bands = []
    for name, (l, h) in bands.items():
        sos = butter(4, [l / (fs / 2), h / (fs / 2)], btype='bandpass', output='sos')
        filt_signal = sosfilt(sos, data, axis=0)

        # Hilbert
        amplitude = np.abs(hilbert(filt_signal, axis=0))

        # Binning (1000 -> 10)
        binned = amplitude.reshape(T, q, n_chans).mean(axis=1)
        data_bands.append(binned)

    # (n_channels, T, n_bands)
    X = np.stack(data_bands, axis=0).transpose(2, 1, 0)

    return X.astype(np.float32)


def run_benchmark(data, n_channels, fs, duration, iterations):
    if data.shape[0] == n_channels:
        data = data.T

    functions = {
        "HGA_LFS": _extract_hga_lfs_direct,
        "Wavelet1": _extract_wavelet1_direct,
        "Wavelet2": _extract_wavelet2_direct,
        "PhysioBand": _extract_physiologicalBand_direct
    }

    print(f"Benchmark: {n_channels} channels, {fs} Hz sampling, {duration} s duration, {iterations} iterations\n")
    header = f"{'Function':<15} | {'Mean (ms)':<12} | {'Std (ms)':<12} | {'Total (s)':<10}"
    print(header)
    print("-" * len(header))

    results_stats = {}
    win_pts = int(duration * fs)
    step_pts = 100  # sliding window step size 100ms = 100 points

    for name, func in functions.items():
        # warmup
        _ = func(data[0:win_pts, :])

        run_times = []
        for i in range(iterations):
            # i=0: [0:1000], i=1: [100:1100], i=2: [200:1200]
            start_idx = i * step_pts
            end_idx = start_idx + win_pts

            if end_idx > data.shape[0]:
                break

            epoch = data[start_idx:end_idx, :]

            start_t = time.perf_counter()
            _ = func(epoch)
            end_t = time.perf_counter()
            run_times.append((end_t - start_t) * 1000)

        times_arr = np.array(run_times)
        avg_ms = np.mean(times_arr)
        std_ms = np.std(times_arr)
        total_s = np.sum(times_arr) / 1000

        results_stats[name] = {"mean": avg_ms, "std": std_ms}
        print(f"{name:<15} | {avg_ms:<12.3f} | {std_ms:<12.3f} | {total_s:<10.4f}")

    return results_stats


if __name__ == "__main__":
    # metadata for different datasets
    datasets = {
        "BCIIV": {
            "subjects": 3,
            "subject_name": ['sub1', 'sub2', 'sub3'],
            "fs_ecog": 1000,
            "fs_dg": 25,
            # file path to extracted features
            "path": 'E:/My projects/finger ECoG/code/HiLoFuseNet/data_preprocessing/preprocessed_data/BCIIV/'
        },
        "Stanford": {
            "subjects": 9,
            "subject_name": ['bp', 'cc', 'ht', 'jc', 'jp', 'mv', 'wc', 'wm', 'zt'],
            "fs_ecog": 1000,
            "fs_dg": 25,
            "path": 'E:/My projects/finger ECoG/code/HiLoFuseNet/data_preprocessing/preprocessed_data/Stanford/'
        },
    }

    # get data from one subject
    fileName = datasets["Stanford"]['path'] + datasets["Stanford"]['subject_name'][2] + '.mat'
    data = loadmat(fileName)
    ECoG = data['data'].T
    fs = 1000
    duration = 1
    iterations = 1000
    _, n_channels = ECoG.shape

    run_benchmark(ECoG, n_channels, fs, duration, iterations)

Benchmark: 64 channels, 1000 Hz sampling, 1 s duration, 1000 iterations

Function        | Mean (ms)    | Std (ms)     | Total (s) 
----------------------------------------------------------
HGA_LFS         | 2.411        | 0.295        | 2.4114    
Wavelet1        | 29.797       | 1.211        | 29.7967   
Wavelet2        | 35.472       | 1.216        | 35.4717   
PhysioBand      | 14.126       | 0.681        | 14.1260   
